In [11]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

import dagshub
dagshub.init(repo_owner="AndriaMakharadze", repo_name="IEEE_Fraud_Detection_AM", mlflow=True)

Initialized MLflow to track repo "AndriaMakharadze/IEEE_Fraud_Detection_AM"

Repository AndriaMakharadze/IEEE_Fraud_Detection_AM initialized!

In [12]:
train_transaction = pd.read_csv("../data/train_transaction.csv")
train_identity = pd.read_csv("../data/train_identity.csv")

df = train_transaction.merge(train_identity, on="TransactionID", how="left")
df = df.drop(columns=["TransactionID"], errors="ignore")

y = df["isFraud"]
X = df.drop(columns=["isFraud"])

X = X.sample(150000, random_state=42)
y = y.loc[X.index]

Cleaning

In [13]:
mlflow.set_experiment("AdaBoost_Training")

with mlflow.start_run(run_name="AdaBoost_Cleaning"):
    null_thresh = 0.8
    cols_to_drop = [c for c in X.columns if X[c].isnull().mean() > null_thresh]
    X = X.drop(columns=cols_to_drop)

    mlflow.log_param("null_threshhold", null_thresh)
    mlflow.log_param("cols_dropped", len(cols_to_drop))
    mlflow.log_metric("cols_remaining", X.shape[1])

    print(f"Dropped {len(cols_to_drop)} high-null columns. Remaining: {X.shape[1]}")

MlflowException: API request to http://127.0.0.1:5000/api/2.0/mlflow/experiments/get-by-name failed with exception HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=AdaBoost_Training (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=5000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))

Feature Engineering

In [ ]:
with mlflow.start_run(run_name="AdaBoost_FeatureEngineering"):
    X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
    X["hour_of_day"] = (X["TransactionDT"] / 3600).astype(int) % 24
    X["null_count"] = X.isnull().sum(axis=1)

    new_features = ["TransactionAmt_log", "hour_of_day", "null_count"]
    mlflow.log_param("new_features", new_features)
    mlflow.log_metric("total_cols_after_eng", X.shape[1])
    print("Feature Engineering Done: ", new_features)

Feature Engineering Done:  ['TransactionAmt_log', 'hour_of_day', 'null_count']
🏃 View run AdaBoost_FeatureEngineering at: http://127.0.0.1:5000/#/experiments/5/runs/15c8fc36ddb24572b9955346bedecdee
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


Feature Selection

In [ ]:
with mlflow.start_run(run_name="AdaBoost_FeatureSelection"):
    num_only = X.select_dtypes(include=["int64", "float64"]).fillna(0)
    corr_matrix = num_only.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]
    X = X.drop(columns=drop_corr, errors="ignore")

    mlflow.log_param("corr_threshold", 0.95)
    mlflow.log_metric("cols_dropped_corr", len(drop_corr))
    mlflow.log_metric("cols_remaining", X.shape[1])
    print(f"Dropped {len(drop_corr)} correlated columns. Remaining: {X.shape[1]}")

Dropped 109 correlated columns. Remaining: 252
🏃 View run AdaBoost_FeatureSelection at: http://127.0.0.1:5000/#/experiments/5/runs/7d4de788f3844237ad63eb2cea630196
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


In [ ]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

num_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])

cat_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))])

preprocessor = ColumnTransformer([("num", num_pipeline, num_cols), ("cat", cat_pipeline, cat_cols)])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

Training

In [ ]:
mlflow.set_experiment("AdaBoost_Training")

with mlflow.start_run(run_name="AdaBoost_Training"):
    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("model", AdaBoostClassifier(
            n_estimators=300,
            random_state=42,
        ))
    ])

    pipeline.fit(X_train, y_train)

    preds = pipeline.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, preds)

    mlflow.log_param("n_estimators", 300)
    mlflow.log_metric("auc", auc)

    mlflow.sklearn.log_model(pipeline, "pipeline_model", registered_model_name="AdaBoost_FraudDetection")

    print("AUC:", auc)

2026/05/04 12:55:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 12:55:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'AdaBoost_FraudDetection'.
2026/05/04 12:55:29 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: AdaBoost_FraudDetection, version 1


AUC: 0.8649372641191515
🏃 View run AdaBoost_Training at: http://127.0.0.1:5000/#/experiments/5/runs/99949de66077480482f7046eed0298a7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


Created version '1' of model 'AdaBoost_FraudDetection'.
